<a href="https://colab.research.google.com/github/robertobautistamx/PSSB/blob/main/Mi_de_PSSB_I_Colab_U1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programación de Sistemas de Base I - Inspección de la Estructura de un Compilador y Procesamiento de Lenguajes
**Asignatura:** Programación de Sistemas de Base I (8.º Semestre)  
**Unidad I:** Introducción a la Compilación  
**Tiempo estimado:** 3 horas (Asíncrono / Guiado)  
**Repositorio Base / Entrega:** Integración con Moodle / GitHub

---


## 1. Objetivos de Aprendizaje
1. Identificar de forma tangible la diferencia entre ejecución compilada e interpretada utilizando las herramientas nativas del entorno (`gcc`, `python3`, `dis`, `ast`).
2. Explorar y visualizar la fase de Análisis Léxico y Sintáctico mediante la inspección del Árbol de Sintaxis Abstracta (AST) y la generación de Bytecode.
3. Comprender el rol de la Tabla de Símbolos en el seguimiento de identificadores durante el proceso de traducción.

---


## 2. Conexión Teórica (NotebookLM)
**Antes de continuar:** Accede al espacio de **NotebookLM del Curso** y realiza las siguientes preguntas de verificación:

- *¿Cuáles son las fases de la etapa de Análisis (Front-End) de un compilador?*  
**RESPUESTA:** La etapa de Análisis (Front-End) se encarga de entender el código que escribes, verificar que no tenga errores y traducirlo a una estructura intermedia
. Sus cuatro fases principales son:
1. Análisis Léxico (Léxer): Lee tu código carácter por carácter, elimina lo innecesario (como espacios en blanco y comentarios)
y agrupa el texto en "palabras" lógicas (lexemas), etiquetándolas bajo una categoría (tokens, como identificador, número u operador)
.

2. Análisis Sintáctico (Párser): Toma esos tokens y verifica que estén ordenados según las reglas gramaticales del lenguaje
. Con ellos, construye un árbol sintáctico que representa la estructura jerárquica de la instrucción
.

3. Análisis Semántico: Revisa el árbol para asegurarse de que las operaciones realmente tengan sentido y coherencia lógica
. Su tarea principal es la verificación de tipos
 (por ejemplo, comprobar que no intentes sumar un texto con un número)
.

4. Generación de Código Intermedio: Traduce el programa a un lenguaje intermedio simplificado (código abstracto)
. Esto permite que esta etapa sea totalmente independiente de la máquina real destino, facilitando que el compilador se adapte después a distintos procesadores

   Elementos de apoyo transversales: Durante todo este proceso actúan de forma simultánea la Tabla de Símbolos (que registra y busca rápidamente los nombres de tus variables, tipos y funciones)
   y el Manejador de Errores (que detecta y te avisa exactamente en qué punto falló el análisis)

- *¿Qué diferencia a un Lexema de un Token?*  
**RESPUESTA:** Un lexema es la secuencia de caracteres concreta (el texto real u ortografía exacta) escrita en el código fuente, como "posicion", "=" o "60"
.
Un token es la categoría abstracta en la que el analizador léxico clasifica a ese lexema para que el analizador sintáctico la procese
. Comúnmente se representa en el formato 〈nombre-token, valor-atributo〉 (por ejemplo, 〈id, apuntador〉 o 〈número, valor〉), donde el nombre es el símbolo abstracto que valida la gramática y el atributo contiene detalles específicos del lexema. En resumen: El lexema es la palabra física escrita en el programa y el token es la etiqueta o rol lógico que el compilador le asigna

- *¿Qué función cumple la Tabla de Símbolos durante la compilación?*  
**RESPUESTA:** La Tabla de Símbolos es una estructura de datos de alto rendimiento (generalmente implementada como una tabla hash) que funciona como un diccionario central durante todo el proceso de traducción
. Su propósito fundamental es servir de puente para transmitir información desde el momento en que declaras una variable, constante o función, hasta cada lugar donde vuelve a utilizarse en el código


---

## 3. Sección Práctica 1: Pipeline de Compilación vs. Interpretación (CLI / Bash)
En esta sección analizaremos cómo el sistema operativo y el entorno procesan un lenguaje compilado (C) frente a uno interpretado/híbrido (Python).

### [CÓDIGO 1.1] Inspección de Herramientas del Sistema
```bash
!echo "=== COMPILADOR C DE LINUX (GCC) ==="
!gcc --version | head -n 1
!echo ""
!echo "=== INTÉRPRETE DE PYTHON ==="
!python3 --version

In [ ]:
!echo "=== COMPILADOR C DE LINUX (GCC) ==="
!gcc --version | head -n 1
!echo ""
!echo "=== INTÉRPRETE DE PYTHON ==="
!python3 --version

=== COMPILADOR C DE LINUX (GCC) ===
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0

=== INTÉRPRETE DE PYTHON ===
Python 3.13.15


### [CÓDIGO 1.2] Creación, Compilación y Análisis de Binario en C
```c
%%writefile hola_compilador.c
#include <stdio.h>
#define inputMsj "Escriba un numero entero: "
#define msj "La suma de %d + %d es: %d\n"

int sumar(int x, int y){
  return x + y;
}

int main() {
    int a = 5;
    int b;
    
    printf(inputMsj);
    scanf("%d",&b);
    int suma = sumar(a, b);
    printf(msj, a, b, suma);
    return 0;
}

In [ ]:
%%writefile hola_compilador.c
# include <stdio.h>
# define inputMsj "Escriba un numero entero: "
# define msj "La suma de %d + %d es: %d\n"

int sumar(int x, int y){
  return x + y;
}

int main() {
    int a = 5;
    int b;

    printf(inputMsj);
    scanf("%d",&b);
    int suma = sumar(a, b);
    printf(msj, a, b, suma);
    return 0;
}

Overwriting hola_compilador.c


## El Preprocesador (Antes del ensamblador)
Esta etapa expande las macros (como `#define`) e incluye los archivos de cabecera (como `#include <stdio.h>`).
```bash
!gcc -E hola_compilador.c -o hola_compilador.i
!echo === CÓDIGO PREPROCESADO ===
!cat hola_compilador.i | tail -n 20


In [ ]:
!gcc -E hola_compilador.c -o hola_compilador.i
!echo === CÓDIGO PREPROCESADO ===
!cat hola_compilador.i | tail -n 20

=== CÓDIGO PREPROCESADO ===
# 2 "hola_compilador.c" 2




# 5 "hola_compilador.c"
int sumar(int x, int y){
  return x + y;
}

int main() {
    int a = 5;
    int b;

    printf("Escriba un numero entero: ");
    scanf("%d",&b);
    int suma = sumar(a, b);
    printf("La suma de %d + %d es: %d\n", a, b, suma);
    return 0;
}


## Representación Intermedia (IR) y Optimizaciones
GCC utiliza una representación intermedia llamada GIMPLE antes de generar el ensamblador. Puedes ver cómo cambia el código antes y después de que el optimizador trabaje.

```bash
!gcc -O2 -fdump-tree-gimple hola_compilador.c -c
!echo === REPRESENTACIÓN INTERMEDIA (GIMPLE) ===
!cat hola_compilador.c.*gimple

In [ ]:
!gcc -O2 -fdump-tree-gimple hola_compilador.c -c
!echo === REPRESENTACIÓN INTERMEDIA (GIMPLE) ===
!cat hola_compilador.c.*gimple

hola_compilador.c: In function ‘main’:
hola_compilador.c:14:5: warning: ignoring return value of ‘scanf’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   14 |     scanf("%d",&b);
      |     ^~~~~~~~~~~~~~
/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `echo === REPRESENTACIÓN INTERMEDIA (GIMPLE) ==='
int sumar (int x, int y)
{
  int D.2555;

  D.2555 = x + y;
  return D.2555;
}


int main ()
{
  int D.2557;

  {
    int a;
    int b;
    int suma;

    try
      {
        a = 5;
        printf ("Escriba un numero entero: ");
        scanf ("%d", &b);
        b.0_1 = b;
        suma = sumar (a, b.0_1);
        b.1_2 = b;
        printf ("La suma de %d + %d es: %d\n", a, b.1_2, suma);
        D.2557 = 0;
        return D.2557;
      }
    finally
      {
        b = {CLOBBER};
      }
  }
  D.2557 = 0;
  return D.2557;
}


__attribute__((artifici

## Compilación generando código ensamblador intermediate (.s)
 El compilador traduce el código **sin optimizaciones** línea por línea directamente a la memoria RAM (la pila o stack)

```bash
!gcc -S hola_compilador.c -o hola_compilador.s
!echo "=== CÓDIGO ENSAMBLADOR GENERADO (SÍNTESIS) ==="
!cat hola_compilador.s | head -n 25

In [ ]:
!gcc -S hola_compilador.c -o hola_compilador.s
!cat hola_compilador.s | head -n 25

cc1: fatal error: hola_compilador.c: No such file or directory
compilation terminated.
cat: hola_compilador.s: No such file or directory


Para producir una traducción del código **optimizada** usa

```bash
!gcc -O3 -S hola_compilador.c -o hola_conmpilador_opt.s
!echo "=== CÓDIGO ENSAMBLADOR OPTIMIZADO GENERADO (SÍNTESIS) ==="
!cat hola_compilador_opt.s | head -n 25

In [ ]:
!gcc -O3 -S hola_compilador.c -o hola_conmpilador_opt.s
!echo "=== CÓDIGO ENSAMBLADOR OPTIMIZADO GENERADO (SÍNTESIS) ==="
!cat hola_compilador_opt.s | head -n 25

hola_compilador.c: In function ‘main’:
hola_compilador.c:14:5: warning: ignoring return value of ‘scanf’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   14 |     scanf("%d",&b);
      |     ^~~~~~~~~~~~~~
=== CÓDIGO ENSAMBLADOR OPTIMIZADO GENERADO (SÍNTESIS) ===
cat: hola_compilador_opt.s: No such file or directory


## Superando limitaciones de la arquitectura fija (X86_64)
La máquina virtual de Colab corre sobre procesadores Intel o AMD de 64 bits.  
**Impacto:** El código ensamblador generado (.s) siempre estará en sintaxis AT&T (por defecto en Linux) y para arquitectura x86_64. Para ver código ensamblador con sintaxis de Intel (más parecida a la de Windows), agregar el parámetro ***-masm=intel*** a la orden de GCC:

```bash
!gcc -S -masm=intel hola_compilador.c -o hola_compilador_8664.s
!echo "=== CÓDIGO ENSAMBLADOR x86_64 GENERADO (SÍNTESIS) ==="
!cat hola_compilador_8664.s | head -n 25


In [ ]:
!gcc -S -masm=intel hola_compilador.c -o hola_compilador_8664.s
!echo "=== CÓDIGO ENSAMBLADOR x86_64 GENERADO (SÍNTESIS) ==="
!cat hola_compilador_8664.s | head -n 25

=== CÓDIGO ENSAMBLADOR x86_64 GENERADO (SÍNTESIS) ===
	.file	"hola_compilador.c"
	.intel_syntax noprefix
	.text
	.globl	sumar
	.type	sumar, @function
sumar:
.LFB0:
	.cfi_startproc
	endbr64
	push	rbp
	.cfi_def_cfa_offset 16
	.cfi_offset 6, -16
	mov	rbp, rsp
	.cfi_def_cfa_register 6
	mov	DWORD PTR -4[rbp], edi
	mov	DWORD PTR -8[rbp], esi
	mov	edx, DWORD PTR -4[rbp]
	mov	eax, DWORD PTR -8[rbp]
	add	eax, edx
	pop	rbp
	.cfi_def_cfa 7, 8
	ret
	.cfi_endproc
.LFE0:
	.size	sumar, .-sumar


## Compilación a código objeto
```bash
# Generación del código objeto (antes de enlazar) y conteo líneas
!gcc -c hola_compilador.c -o hola_compilador.o
!objdump -d hola_compilador.o | wc -l

In [ ]:
# Generación del código objeto (antes de enlazar) y conteo líneas
!gcc -c hola_compilador.c -o hola_compilador.o
!objdump -d hola_compilador.o | wc -l

60


## Compilación a código máquina ejecutable

```bash
# Generación del código ejecutable final (después de enlazar) y conteo líneas
!gcc hola_compilador.c -o hola_compilador
!objdump -d hola_compilador | wc -l


In [ ]:
# Generación del código ejecutable final (después de enlazar) y conteo líneas
!gcc hola_compilador.c -o hola_compilador
!objdump -d hola_compilador | wc -l

194


## El "Monstruo Completo" (Enlace Estático)  
Por defecto, GCC usa ***enlace dinámico*** usando una estructura de datos tabular, la tabla **PLT** (*Procedure Linkage Table*) ya que el código usa `printf`, el programa necesita conectarse con la biblioteca estándar de C (`libc.so`). El enlazador no copia el código de `printf` dentro del archivo para no duplicar espacio en el disco; en su lugar, crea un "trampolín" o acceso directo en una sección llamada `<printf@plt>`.

Cada vez que se invoca a `printf`, realmente salta a este bloque plt, el cual averigua en qué parte de la memoria RAM del sistema operativo se encuentra la verdadera función `printf` y redirige el control allí en tiempo de ejecución.

Para ver cómo el enlazador inyecta literalmente miles y miles de líneas copiando físicamente el código de `printf` y todas sus dependencias dentro del ejecutable, se debe compilar de forma estática:

```bash
# Compilación estática
!gcc -static hola_compilador.c -o hola_estatico

# Conteo de líneas de código ensamblador
!objdump -d hola_estatico | wc -l


In [ ]:
# Compilación estática
!gcc -static hola_compilador.c -o hola_estatico

# Conteo de líneas de código ensamblador
!objdump -d hola_estatico | wc -l

182934


## Ejecución del código
```bash
!echo ""
!echo "=== EJECUCIÓN DEL PROGRAMA OBJETO ==="
!./hola_compilador


In [ ]:
!echo ""
!echo "=== EJECUCIÓN DEL PROGRAMA OBJETO ==="
!./hola_compilador


=== EJECUCIÓN DEL PROGRAMA OBJETO ===
Escriba un numero entero: 5
La suma de 5 + 5 es: 10


## 4. Sección Práctica 2: Revelando el Front-End (Análisis Léxico y AST)
Para comprender cómo el compilador "desarma" el código fuente en tokens y estructuras jerárquicas, inspeccionaremos el compilador de Python a nivel interno.

### [CÓDIGO 2.1] Inspección del Analizador Léxico (Tokenizador)

```python
import token
import tokenize
from io import BytesIO

# Código fuente de prueba en formato cadena
codigo_fuente = "suma = a + 10"

# Tokenización del código fuente
tokens = tokenize.tokenize(BytesIO(codigo_fuente.encode("utf-8")).readline)

print(
    f"{'LINEA/COL':<12} | {'TIPO DE TOKEN':<20} | {'VALOR (LEXEMA)':<15}"
)
print("-" * 55)
for tok in tokens:
    if tok.type in (
        tokenize.ENCODING,
        tokenize.ENDMARKER,
        tokenize.NL,
        tokenize.NEWLINE,
    ):
        continue
    nombre_token = token.tok_name[tok.type]
    posicion = f"{tok.start[0]}:{tok.start[1]}"
    print(f"{posicion:<12} | {nombre_token:<20} | {tok.string:<15}")


In [ ]:
import token
import tokenize
from io import BytesIO

# Código fuente de prueba en formato cadena
codigo_fuente = "suma = a + 10"

# Tokenización del código fuente
tokens = tokenize.tokenize(BytesIO(codigo_fuente.encode("utf-8")).readline)

print(
    f"{'LINEA/COL':<12} | {'TIPO DE TOKEN':<20} | {'VALOR (LEXEMA)':<15}"
)
print("-" * 55)
for tok in tokens:
    if tok.type in (
        tokenize.ENCODING,
        tokenize.ENDMARKER,
        tokenize.NL,
        tokenize.NEWLINE,
    ):
        continue
    nombre_token = token.tok_name[tok.type]
    posicion = f"{tok.start[0]}:{tok.start[1]}"
    print(f"{posicion:<12} | {nombre_token:<20} | {tok.string:<15}")

LINEA/COL    | TIPO DE TOKEN        | VALOR (LEXEMA) 
-------------------------------------------------------
1:0          | NAME                 | suma           
1:5          | OP                   | =              
1:7          | NAME                 | a              
1:9          | OP                   | +              
1:11         | NUMBER               | 10             


### [CÓDIGO 2.2] Construcción del Árbol de Sintaxis Abstracta (AST)

```python
import ast

# Generar y visualizar la estructura sintáctica
arbol = ast.parse("suma = a + 10")
print("=== ÁRBOLES DE SINTAXIS ABSTRACTA (REPRESENTACIÓN EN TEXTO) ===")
print(ast.dump(arbol, indent=4))


In [ ]:
import ast

# Generar y visualizar la estructura sintáctica
arbol = ast.parse("suma = a + 10")
print("=== ÁRBOLES DE SINTAXIS ABSTRACTA (REPRESENTACIÓN EN TEXTO) ===")
print(ast.dump(arbol, indent=4))

=== ÁRBOLES DE SINTAXIS ABSTRACTA (REPRESENTACIÓN EN TEXTO) ===
Module(
    body=[
        Assign(
            targets=[
                Name(id='suma', ctx=Store())],
            value=BinOp(
                left=Name(id='a', ctx=Load()),
                op=Add(),
                right=Constant(value=10)))])


## 5. Sección Práctica 3: Inspección de la Tabla de Símbolos y Bytecode
En esta sección simularemos la interacción con la Tabla de Símbolos y observaremos la generación de código intermedio.

### [CÓDIGO 3.1] Simulación de una Tabla de Símbolos básica en Python
```python
class TablaDeSimbolos:

    def __init__(self):
        self.simbolos = {}

    def insertar(self, nombre, tipo, valor=None, ambito="global"):
        if nombre in self.simbolos:
            print(f"[ERROR LÉXICO/SINTÁCTICO] Identificador '{nombre}' ya declarado.")
        else:
            self.simbolos[nombre] = {
                "tipo": tipo,
                "valor": valor,
                "ambito": ambito,
            }
            print(f"[TABLA DE SÍMBOLOS] Insertado: {nombre} ({tipo})")

    def buscar(self, nombre):
        return self.simbolos.get(nombre, None)

    def mostrar(self):
        print("\n=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===")
        print(f"{'NOMBRE':<12} | {'TIPO':<10} | {'ÁMBITO':<10} | {'VALOR':<10}")
        print("-" * 50)
        for nombre, datos in self.simbolos.items():
            print(
                f"{nombre:<12} | {datos['tipo']:<10} | {datos['ambito']:<10} | {str(datos['valor']):<10}"
            )


# Prueba de la Tabla de Símbolos
ts = TablaDeSimbolos()
ts.insertar("a", "ENTERO", 5)
ts.insertar("b", "ENTERO", 10)
ts.insertar("suma", "ENTERO", 15)
ts.mostrar()
```


In [ ]:
class TablaDeSimbolos:

    def __init__(self):
        self.simbolos = {}

    def insertar(self, nombre, tipo, valor=None, ambito="global"):
        if nombre in self.simbolos:
            print(f"[ERROR LÉXICO/SINTÁCTICO] Identificador '{nombre}' ya declarado.")
        else:
            self.simbolos[nombre] = {
                "tipo": tipo,
                "valor": valor,
                "ambito": ambito,
            }
            print(f"[TABLA DE SÍMBOLOS] Insertado: {nombre} ({tipo})")

    def buscar(self, nombre):
        return self.simbolos.get(nombre, None)

    def mostrar(self):
        print("\n=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===")
        print(f"{'NOMBRE':<12} | {'TIPO':<10} | {'ÁMBITO':<10} | {'VALOR':<10}")
        print("-" * 50)
        for nombre, datos in self.simbolos.items():
            print(
                f"{nombre:<12} | {datos['tipo']:<10} | {datos['ambito']:<10} | {str(datos['valor']):<10}"
            )


# Prueba de la Tabla de Símbolos
ts = TablaDeSimbolos()
ts.insertar("a", "ENTERO", 5)
ts.insertar("b", "ENTERO", 10)
ts.insertar("suma", "ENTERO", 15)
ts.mostrar()

[TABLA DE SÍMBOLOS] Insertado: a (ENTERO)
[TABLA DE SÍMBOLOS] Insertado: b (ENTERO)
[TABLA DE SÍMBOLOS] Insertado: suma (ENTERO)

=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===
NOMBRE       | TIPO       | ÁMBITO     | VALOR     
--------------------------------------------------
a            | ENTERO     | global     | 5         
b            | ENTERO     | global     | 10        
suma         | ENTERO     | global     | 15        


### [CÓDIGO 3.2] Desensamblado a Código Intermedio / Bytecode (Máquina de Pila)
```python
import dis

def calcular():
    a = 5
    b = 10
    suma = a + b
    return suma


print("=== BYTECODE GENERADO PARA LA MÁQUINA VIRTUAL DE PYTHON ===")
dis.dis(calcular)
```

In [ ]:
import dis

def calcular():
    a = 5
    b = 10
    suma = a + b
    return suma


print("=== BYTECODE GENERADO PARA LA MÁQUINA VIRTUAL DE PYTHON ===")
dis.dis(calcular)

=== BYTECODE GENERADO PARA LA MÁQUINA VIRTUAL DE PYTHON ===
  3           RESUME                   0

  4           LOAD_CONST               1 (5)
              STORE_FAST               0 (a)

  5           LOAD_CONST               2 (10)
              STORE_FAST               1 (b)

  6           LOAD_FAST_LOAD_FAST      1 (a, b)
              BINARY_OP                0 (+)
              STORE_FAST               2 (suma)

  7           LOAD_FAST                2 (suma)
              RETURN_VALUE


## 6. Desafío Asíncrono / Entregable de la Unidad I (Avance del Producto Integrador)

### Contexto del Proyecto Integrador Autónomo:
Durante el semestre, cada equipo concebirá, diseñará e implementará un **Analizador Léxico y Sintáctico (Compiler Front-End)** para un lenguaje original propuesto por el propio equipo.

Ejemplos de proyectos de semestres anteriores:
- **DSL para Configuración de Robots / Drones:** Lenguaje de comandos simples (`FORWARD 10`, `ROTATE 90`).
- **Lenguaje de Consultas para Grafos/Tablas:** Alternativa simplificada a SQL (`SELECT age FROM users WHERE status == 1`).
- **Mini-Lenguaje Matemático / Scripting:** Soporte para vectores, matrices o evaluación de expresiones lógicas/aritméticas.
- **Lenguaje Formato/Marcado Personalizado:** Generador de reportes en HTML/Markdown a partir de una sintaxis limpia.

---

### Instrucciones del Desafío U1 (Fase 0: Definición e Inspección Inicial):

#### Parte A: Documento de Especificación del Lenguaje (RFC del Equipo)
Crea un archivo `ESPECIFICACION.docx` que contenga:
1. **Nombre del Lenguaje Original y Propósito:** ¿Qué problema resuelve o a quién va dirigido?
2. **Ejemplo de Código Fuente Válido:** Muestra un fragmento de código de al menos 10-15 líneas escrito en tu nuevo lenguaje.
3. **Catálogo Preliminar de Tokens:** Lista las palabras reservadas, identificadores, constantes (enteras, flotantes, cadenas) y operadores que usará tu lenguaje.

#### Parte B: Prototipo de Inspección en Código (Colab Execution)
Utilizando las celdas mágicas `%%writefile` (en Python o C):
1. Crea un archivo con una cadena que contenga tu ejemplo de código fuente original.
2. Implementa una función de prueba preliminar (o utiliza la librería `tokenize` de Python como simulador) que tome la cadena de tu lenguaje y la desglose en una lista de componentes léxicos.
3. Muestra una estructura en código que sirva como **Tabla de Símbolos inicial** donde se registren las variables declaradas en tu lenguaje.

---

### Criterios Mínimos que Debe Cumplir Cualquier Lenguaje Propuesto (Checklist de Viabilidad):
Para que la propuesta sea aprobada por el catedrático, el lenguaje debe cumplir con:
- [ ] Poseer al menos **3 tipos de tokens bien diferenciados** (ej. Palabras Reservadas, Identificadores, Literales).
- [ ] Incluir soporte para **expresiones aritméticas o lógicas** anidadas.
- [ ] Incluir al menos una **estructura de control de flujo** (ej. `if/else`, `while`, `repeat`) o una **estructura de bloques** (funciones, comandos).
- [ ] Definir un mecanismo explícito de asignación o declaración de datos.

---

### Autoevaluación Muestreada (Comprobación Tipo Gradiance)
Responde las siguientes preguntas analizando la sintaxis de tu nuevo lenguaje y valida tus razonamientos en NotebookLM:

1. **Pregunta 1 (Conflictos Léxicos):** Al revisar las palabras reservadas y los identificadores de tu lenguaje, ¿existe alguna regla léxica que pudiera causar ambigüedad (por ejemplo, que una palabra reservada coincida con el patrón de un identificador de usuario)? ¿Cómo la resolverá tu analizador?
2. **Pregunta 2 (Estructura de la Tabla de Símbolos):** De los elementos de tu lenguaje original, ¿cuáles atributos (tipo, valor, ámbito, dirección de memoria) necesitará almacenar tu Tabla de Símbolos cuando se procese una declaración?

---

### Formato de Entrega / Portafolio de Evidencias
1. Guarda este cuaderno con todas las salidas ejecutadas (`Archivo` -> `Guardar una copia en GitHub` / `PEREZ JUAN PSSB I Colab U1.ipynb`).
2. Guarda el documento `ESPECIFICACION.docx` con la especificación de tu lenguaje en tu repositorio personal (no repositorios grupales) de GitHub.
3. Registra en **Moodle** el enlace del cuaderno ejecutable en Colab y el de tu repositorio conteniendo la especificación formal del proyecto.

## Sección de trabajo no dirigido
Prototipo de Inspección en Código (Colab Execution)
Utilizando las celdas mágicas `%%writefile` (en Python o C):
1. Crea un archivo con una cadena que contenga tu ejemplo de código fuente original.

In [ ]:
%%writefile homescript_ejemplo.txt
DISPOSITIVO clima
DISPOSITIVO ventilador
DISPOSITIVO luces
SENSOR temperatura
SENSOR humedad

PARAMETRO frio = 18
PARAMETRO calor = 28
PARAMETRO humedad_max = 60.5

SI temperatura >= calor Y humedad > humedad_max {
    ENCENDER clima
    ENCENDER ventilador
} SINO SI temperatura <= frio {
    APAGAR clima
    APAGAR ventilador
}

MIENTRAS temperatura > frio {
    ENCENDER luces
}

ESTADO Noche {
    CUANDO temperatura < frio -> Frio
}

Writing homescript_ejemplo.txt


2. Implementa una función de prueba preliminar (o utiliza la librería `tokenize` de Python como simulador) que tome la cadena de tu lenguaje y la desglose en una lista de componentes léxicos.

In [ ]:
import re

RESERVADAS = {
    "DISPOSITIVO", "SENSOR", "PARAMETRO", "SI", "SINO", "MIENTRAS",
    "ENCENDER", "APAGAR", "ESTADO", "CUANDO", "Y", "O", "NO",
    "VERDADERO", "FALSO"
}

TOKENS = [
    ("FLOTANTE", r"\d+\.\d+"), ("ENTERO", r"\d+"),
    ("CADENA", r'"[^"\n]*"'), ("FLECHA", r"->"),
    ("REL_OP", r">=|<=|==|!=|>|<"), ("ARIT_OP", r"[+\-*/]"),
    ("ASIGNACION", r"="), ("LLAVE_ABRE", r"\{"), ("LLAVE_CIERRA", r"\}"),
    ("IDENTIFICADOR", r"[A-Za-z_][A-Za-z0-9_]*"),
    ("ESPACIO", r"[ \t]+"), ("SALTO_LINEA", r"\n"), ("DESCONOCIDO", r".")
]

REGEX = "|".join(f"(?P<{n}>{p})" for n, p in TOKENS)

def tokenizar(codigo):
    tokens, linea = [], 1

    for m in re.finditer(REGEX, codigo):
        tipo, valor = m.lastgroup, m.group()

        if tipo == "SALTO_LINEA":
            linea += 1
        elif tipo != "ESPACIO":
            if tipo == "IDENTIFICADOR" and valor in RESERVADAS:
                tipo = "PALABRA_RESERVADA"
            tokens.append({"tipo": tipo, "valor": valor, "linea": linea})

    return tokens

with open("homescript_ejemplo.txt", encoding="utf-8") as f:
    tokens = tokenizar(f.read())

for token in tokens:
    print(token)

{'tipo': 'PALABRA_RESERVADA', 'valor': 'DISPOSITIVO', 'linea': 1}
{'tipo': 'IDENTIFICADOR', 'valor': 'clima', 'linea': 1}
{'tipo': 'PALABRA_RESERVADA', 'valor': 'DISPOSITIVO', 'linea': 2}
{'tipo': 'IDENTIFICADOR', 'valor': 'ventilador', 'linea': 2}
{'tipo': 'PALABRA_RESERVADA', 'valor': 'DISPOSITIVO', 'linea': 3}
{'tipo': 'IDENTIFICADOR', 'valor': 'luces', 'linea': 3}
{'tipo': 'PALABRA_RESERVADA', 'valor': 'SENSOR', 'linea': 4}
{'tipo': 'IDENTIFICADOR', 'valor': 'temperatura', 'linea': 4}
{'tipo': 'PALABRA_RESERVADA', 'valor': 'SENSOR', 'linea': 5}
{'tipo': 'IDENTIFICADOR', 'valor': 'humedad', 'linea': 5}
{'tipo': 'PALABRA_RESERVADA', 'valor': 'PARAMETRO', 'linea': 7}
{'tipo': 'IDENTIFICADOR', 'valor': 'frio', 'linea': 7}
{'tipo': 'ASIGNACION', 'valor': '=', 'linea': 7}
{'tipo': 'ENTERO', 'valor': '18', 'linea': 7}
{'tipo': 'PALABRA_RESERVADA', 'valor': 'PARAMETRO', 'linea': 8}
{'tipo': 'IDENTIFICADOR', 'valor': 'calor', 'linea': 8}
{'tipo': 'ASIGNACION', 'valor': '=', 'linea': 8}
{'ti

3. Muestra una estructura en código que sirva como **Tabla de Símbolos inicial** donde se registren las variables declaradas en tu lenguaje.

In [ ]:
tabla_simbolos = {}

def registrar_simbolo(tokens):
    for i, token in enumerate(tokens):
        if token["tipo"] == "PALABRA_RESERVADA" and token["valor"] in ("DISPOSITIVO", "SENSOR", "PARAMETRO"):
            categoria = token["valor"]
            nombre = tokens[i + 1]["valor"]

            entrada = {
                "categoria": categoria,
                "tipo_dato": "entidad",
                "valor_inicial": None,
                "ambito": "global",
                "linea_declaracion": token["linea"]
            }

            if (categoria == "PARAMETRO" and i + 3 < len(tokens)
                    and tokens[i + 2]["tipo"] == "ASIGNACION"):
                valor = tokens[i + 3]
                entrada["valor_inicial"] = valor["valor"]
                entrada["tipo_dato"] = "flotante" if valor["tipo"] == "FLOTANTE" else "entero"

            tabla_simbolos[nombre] = entrada

registrar_simbolo(tokens)

for nombre, datos in tabla_simbolos.items():
    print(nombre, "->", datos)

clima -> {'categoria': 'DISPOSITIVO', 'tipo_dato': 'entidad', 'valor_inicial': None, 'ambito': 'global', 'linea_declaracion': 1}
ventilador -> {'categoria': 'DISPOSITIVO', 'tipo_dato': 'entidad', 'valor_inicial': None, 'ambito': 'global', 'linea_declaracion': 2}
luces -> {'categoria': 'DISPOSITIVO', 'tipo_dato': 'entidad', 'valor_inicial': None, 'ambito': 'global', 'linea_declaracion': 3}
temperatura -> {'categoria': 'SENSOR', 'tipo_dato': 'entidad', 'valor_inicial': None, 'ambito': 'global', 'linea_declaracion': 4}
humedad -> {'categoria': 'SENSOR', 'tipo_dato': 'entidad', 'valor_inicial': None, 'ambito': 'global', 'linea_declaracion': 5}
frio -> {'categoria': 'PARAMETRO', 'tipo_dato': 'entero', 'valor_inicial': '18', 'ambito': 'global', 'linea_declaracion': 7}
calor -> {'categoria': 'PARAMETRO', 'tipo_dato': 'entero', 'valor_inicial': '28', 'ambito': 'global', 'linea_declaracion': 8}
humedad_max -> {'categoria': 'PARAMETRO', 'tipo_dato': 'flotante', 'valor_inicial': '60.5', 'ambito':

1. **Pregunta 1 (Conflictos Léxicos):** Al revisar las palabras reservadas y los identificadores de tu lenguaje, ¿existe alguna regla léxica que pudiera causar ambigüedad (por ejemplo, que una palabra reservada coincida con el patrón de un identificador de usuario)? ¿Cómo la resolverá tu analizador?

**RESPUESTA:** No hay ambigüedad porque todas las palabras reservadas se escriben en mayúsculas, mientras que los identificadores de usuario usan minúsculas o PascalCase. Aun así, el analizador no depende del estilo: primero reconoce cualquier cadena que cumpla el patrón [A-Za-z_][A-Za-z0-9_]* como identificador y, después, verifica si coincide exactamente con el conjunto cerrado de palabras reservadas; si coincide, se reclasifica como PALABRA_RESERVADA, y si no, se conserva como IDENTIFICADOR. Esto se resuelve con un simple lookup posterior, sin reglas adicionales, tal como se implementa en el tokenizador de la Celda 2.

2. **Pregunta 2 (Estructura de la Tabla de Símbolos):** De los elementos de tu lenguaje original, ¿cuáles atributos (tipo, valor, ámbito, dirección de memoria) necesitará almacenar tu Tabla de Símbolos cuando se procese una declaración?

**RESPUESTA:**Por cada identificador declarado (dispositivo, sensor o parámetro) se almacenan: nombre (el lexema tal cual aparece en el código), categoría (DISPOSITIVO, SENSOR o PARAMETRO), tipo de dato (entidad para dispositivos/sensores, o entero/flotante/cadena para parámetros, inferido del literal asignado), valor inicial (el valor asignado, o nulo si no aplica), ámbito (por ahora global, ya que el lenguaje aún no define funciones ni variables locales) y línea de declaración (útil para depuración y mensajes de error). No se incluye dirección de memoria porque HomeScript aún no compila a un formato binario/ejecutable de bajo nivel.

